# Kiểm chứng BERTScore: JD requirement vs CV bullet

Câu hỏi cần kiểm tra: BERTScore có xếp hạng độ khớp giữa dòng yêu cầu JD và dòng CV **cùng thứ tự với nhãn người chấm** hay không.

> Mọi con số đưa vào báo cáo phải lấy từ lần chạy thật. Nhãn trong `pairs.csv` mẫu là nhãn tổng hợp, chỉ dùng để chạy thử quy trình.

**Trước khi chạy:** bật GPU (`Runtime` → `Change runtime type` → T4).

## 1. Cài đặt

In [ ]:
!pip install -q bert-score scipy pandas

## 2. Tải dữ liệu
Chạy ô dưới rồi chọn file `pairs.csv` (4 cột: `pair_id`, `jd_requirement`, `cv_bullet`, `human_label`).

In [ ]:
import os
if not os.path.exists("pairs.csv"):
    from google.colab import files
    files.upload()

import pandas as pd
df = pd.read_csv("pairs.csv")
print(df.shape)
print(df["human_label"].value_counts().sort_index())
df.head()

In [ ]:
# Kiểm tra dữ liệu hợp lệ
assert {"pair_id","jd_requirement","cv_bullet","human_label"} <= set(df.columns), "Thiếu cột"
assert df["human_label"].isin([0,1,2]).all(), "Nhãn phải là 0, 1 hoặc 2"
assert df[["jd_requirement","cv_bullet"]].notna().all().all(), "Có ô trống"
if len(df) < 30:
    print("Cảnh báo: dưới 30 cặp thì kết luận rất yếu.")

## 3. Khởi tạo BERTScorer
Lần đầu sẽ tải `xlm-roberta-large` (vài GB). Dữ liệu mẫu bằng tiếng Anh nên dùng `lang="en"`; nếu dữ liệu của bạn là tiếng Việt thì đổi thành `"vi"`.

In [ ]:
from bert_score import BERTScorer

LANG = "en"   # đổi thành "vi" nếu dữ liệu tiếng Việt
scorer = BERTScorer(model_type="xlm-roberta-large", lang=LANG)

## 4. Thử nhanh (kiểm tra môi trường và các trường hợp bẫy)

In [ ]:
def show(jd, cvs):
    _, R, _ = scorer.score(cands=cvs, refs=[jd] * len(cvs))
    print("JD:", jd)
    for c, r in zip(cvs, R):
        print(f"  {float(r):.3f}  {c}")
    print()

# Skill gần giống về chữ
show("Requires strong Java skills",
     ["Has experience programming in Java",
      "Has experience programming in JavaScript",
      "Has experience programming in Python"])

# Số năm kinh nghiệm
show("Requires 3 years of Java experience",
     ["Has 3 years of Java experience",
      "Has 8 months of Java experience",
      "Has 10 years of Java experience",
      "Has no Java experience"])

## 5. Chấm điểm toàn bộ dữ liệu
`cands` là dòng CV, `refs` là dòng JD. **Recall** là cột chính.

In [ ]:
import re

def mask_numbers(text):
    # Thay số bằng ký hiệu chung. Lưu ý: cũng che cả số phiên bản như "Java 8".
    return re.sub(r"\d+(?:[.,]\d+)?", "N", str(text))

def run(cands, refs, batch_size=32):
    P, R, F1 = scorer.score(cands=list(cands), refs=list(refs), batch_size=batch_size)
    return R.tolist(), P.tolist(), F1.tolist()

# Cách 1: văn bản gốc
df["recall"], df["precision"], df["f1"] = run(df["cv_bullet"], df["jd_requirement"])

# Cách 2: đã che số
df["recall_masked"], _, _ = run(
    df["cv_bullet"].map(mask_numbers),
    df["jd_requirement"].map(mask_numbers),
)

df.to_csv("pairs_scored.csv", index=False)
df.head()

## 6. Kiểm chứng
### 6.1 Điểm trung bình theo nhãn người chấm
Điều cần xem: điểm có tăng dần theo nhãn (0 thấp, 1 vừa, 2 cao) và các nhóm có tách nhau rõ hay không.

In [ ]:
summary = df.groupby("human_label")[["recall", "recall_masked"]].agg(["mean", "std", "count"])
summary.round(4)

### 6.2 Tương quan hạng Spearman

In [ ]:
from scipy.stats import spearmanr

for col in ["recall", "recall_masked", "f1"]:
    rho, p = spearmanr(df[col], df["human_label"])
    print(f"{col:15s} Spearman = {rho:.3f}   (p = {p:.4f}, n = {len(df)})")

## 7. Phân tích bổ sung (không bắt buộc)

In [ ]:
import matplotlib.pyplot as plt

# Phân bố điểm recall theo nhãn
ax = df.boxplot(column="recall", by="human_label", figsize=(6, 4))
plt.suptitle("")
plt.title("BERTScore Recall theo nhãn người chấm")
plt.xlabel("human_label"); plt.ylabel("recall")
plt.show()

In [ ]:
# Với mỗi dòng JD: BERTScore có xếp cặp nhãn cao nhất lên đầu không?
hit = []
for jd, g in df.groupby("jd_requirement"):
    if g["human_label"].nunique() < 2:
        continue
    top = g.loc[g["recall"].idxmax(), "human_label"]
    hit.append(top == g["human_label"].max())
print(f"Top-1 đúng: {sum(hit)}/{len(hit)} = {sum(hit)/len(hit):.1%} (số JD có đủ nhiều mức nhãn)")

In [ ]:
# Các cặp nhãn 0 mà điểm cao nhất, và các cặp nhãn 2 mà điểm thấp nhất
cols = ["jd_requirement", "cv_bullet", "human_label", "recall"]
print("Nhãn 0 nhưng điểm cao (BERTScore bị đánh lừa):")
display(df[df.human_label == 0].nlargest(8, "recall")[cols])
print("Nhãn 2 nhưng điểm thấp (BERTScore bỏ sót):")
display(df[df.human_label == 2].nsmallest(8, "recall")[cols])

In [ ]:
# Tải kết quả về máy
from google.colab import files
files.download("pairs_scored.csv")

## 8. Mẫu câu cho báo cáo (điền số thật)

> Nhóm chạy BERTScore (`xlm-roberta-large`, Recall) trên **[N]** cặp (dòng yêu cầu JD, dòng CV), nhãn **[cách gán, số người chấm]**. Điểm trung bình theo nhãn 0, 1, 2 lần lượt là **[...]**, **[...]**, **[...]**. Spearman giữa điểm và nhãn là **[...]** (n = **[N]**); sau khi che số là **[...]**.
>
> Hạn chế: số cặp nhỏ, dữ liệu tổng hợp, nhãn do nhóm gán, kết quả chỉ mang tính kiểm tra ban đầu.

**Lỗi thường gặp:** hết bộ nhớ GPU thì giảm `batch_size` xuống 8 hoặc 16; mỗi dòng nên là một câu hoặc một gạch đầu dòng (giới hạn khoảng 512 token).